# Stage 3 — Attribution patching (the efficient method)

**Capstone: Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2**

In Stage 2 you measured head importance the trusted-but-slow way: switch each of the 144 heads in turn and re-run the model (144 forward passes). This stage does it the **fast** way.

**Plain-English idea.** Instead of re-cooking the dish 144 times to see which ingredient matters, we taste it once and use a bit of calculus to *estimate* how much each ingredient contributes — all at once. That estimate uses one forward pass and one backward pass, no matter how many components there are. This is **attribution patching** (Nanda, 2023): the cheap, gradient-based stand-in for activation patching, and a clean example of the kind of efficient attribution method the LM Transparency Tool is built on.

The output is a head-importance matrix in the **same shape and scale** as Stage 2, so Stage 4 can compare them directly. Because it is only an *estimate*, the numbers will not match Stage 2 exactly — measuring how closely they agree is the whole point of the project.

*Runs in Google Colab; GPU recommended. Tools: TransformerLens (Nanda & Bloom, 2022); method from Nanda (2023).*

## 1. Setup and the same IOI data as Stage 2

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch, itertools, random, time
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)
n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
print(f"GPT-2 small on {device}: {n_layers} layers x {n_heads} heads")

In [ ]:
template = "When{A} and{B} went to the shop,{S} gave a drink to"
candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]
names = [n for n in candidate_names if model.to_tokens(n, prepend_bos=False).shape[1] == 1]

random.seed(0)
pairs = [(a, b) for a, b in itertools.permutations(names, 2)]
random.shuffle(pairs)
pairs = pairs[:15]

clean_prompts     = [template.format(A=a, B=b, S=b) for a, b in pairs]
corrupted_prompts = [template.format(A=a, B=b, S=a) for a, b in pairs]
clean_tokens     = model.to_tokens(clean_prompts)
corrupted_tokens = model.to_tokens(corrupted_prompts)

N = len(pairs)
rows = torch.arange(N, device=device)
io_tokens = torch.tensor([model.to_single_token(a) for a, b in pairs], device=device)
s_tokens  = torch.tensor([model.to_single_token(b) for a, b in pairs], device=device)

def logit_diff_tensor(logits):
    final = logits[:, -1, :]
    return (final[rows, io_tokens] - final[rows, s_tokens]).mean()   # keep as a tensor for backprop

def logit_diff(logits):
    return logit_diff_tensor(logits).item()

## 2. The two baselines (needed for the scale)
Same as before: clean should be clearly positive, corrupted negative. We reuse `CLEAN_LD − CORRUPT_LD` as the scale so this matrix matches Stage 2.

In [ ]:
with torch.no_grad():
    CLEAN_LD = logit_diff(model(clean_tokens))
    CORRUPT_LD = logit_diff(model(corrupted_tokens))
denom = CLEAN_LD - CORRUPT_LD
print(f"Clean:     {CLEAN_LD:+.3f}")
print(f"Corrupted: {CORRUPT_LD:+.3f}")
print(f"Scale (clean - corrupted): {denom:.3f}")

## 3. Attribution patching in three passes
We need three things per head:
1. its output on the **clean** sentence,
2. its output on the **corrupted** sentence, and
3. the **gradient** of the logit difference with respect to that output on the corrupted run.

The importance estimate for each head is then:

`(clean_output − corrupted_output) · gradient`

summed over positions. In words: *how far the head's activity would move if we made it clean, multiplied by how much the answer cares about that activity.* Big product = important head. We grab the gradients using a **backward hook**.

In [ ]:
filter_z = lambda name: name.endswith("hook_z")

# (1) clean head outputs, no gradient needed
with torch.no_grad():
    _, clean_cache = model.run_with_cache(clean_tokens, names_filter=filter_z)

# (2) + (3) corrupted head outputs AND their gradients, via forward + backward hooks
corrupt_acts, corrupt_grads = {}, {}
def save_act(act, hook):  corrupt_acts[hook.name] = act.detach()
def save_grad(grad, hook): corrupt_grads[hook.name] = grad.detach()

model.reset_hooks()
for l in range(n_layers):
    name = utils.get_act_name("z", l)
    model.add_hook(name, save_act, "fwd")
    model.add_hook(name, save_grad, "bwd")

torch.set_grad_enabled(True)
t0 = time.time()
metric = logit_diff_tensor(model(corrupted_tokens))
metric.backward()
attribution_time = time.time() - t0
model.reset_hooks()
torch.set_grad_enabled(False)
print(f"Forward + backward done in {attribution_time:.3f} s")

In [ ]:
# Combine into a per-head importance matrix on the same scale as Stage 2
attribution = np.zeros((n_layers, n_heads))
for l in range(n_layers):
    name = utils.get_act_name("z", l)
    contrib = (clean_cache[name] - corrupt_acts[name]) * corrupt_grads[name]  # [batch, pos, head, d_head]
    per_head = contrib.sum(dim=(0, 1, 3))                                     # -> [head]
    attribution[l] = (per_head / denom).cpu().numpy()

print("Attribution importance matrix ready.")

In [ ]:
plt.figure(figsize=(8, 6))
lim = abs(attribution).max()
plt.imshow(attribution, cmap="RdBu", vmin=-lim, vmax=lim)
plt.colorbar(label="attribution importance (estimated restoration)")
plt.xlabel("head"); plt.ylabel("layer"); plt.title("Attribution-patching importance per head")
plt.xticks(range(n_heads)); plt.yticks(range(n_layers))
plt.show()

flat = [(attribution[l, h], l, h) for l in range(n_layers) for h in range(n_heads)]
flat.sort(reverse=True)
print("Top 10 heads identified by attribution patching:")
for score, l, h in flat[:10]:
    print(f"  layer {l:2d}, head {h:2d}   attribution = {score:.3f}")

## 4. Compare by eye, and note the efficiency
Put this heatmap next to your Stage 2 one. The same regions should light up — the S-inhibition, name-mover and induction heads — even though the exact numbers differ, because this is an approximation.

**Efficiency (a Stage-3 result for RQ3):** attribution patching produced importances for **all** heads from **one forward pass plus one backward pass**. Activation patching in Stage 2 needed **one forward pass per head** — 144 of them. That gap is exactly the efficiency advantage the project is examining.

In [ ]:
print(f"Attribution patching: 1 forward + 1 backward pass  ({attribution_time:.3f} s here)")
print(f"Activation patching:  {n_layers*n_heads} forward passes (one per head)")

## What you've built
You now have the **second method's** importance matrix, on the same scale as the first. That is both methods in hand.

**Next — Stage 4 (the payoff):** compute both matrices together and answer the research questions —
- **Agreement:** how well do the two rankings match each other, and the known IOI circuit? (rank correlation, overlap, precision/recall)
- **Faithfulness:** using the Stage 2 ablation harness, ablate the top-k heads from each method and plot how the behaviour falls away — the faithfulness curves.

Save this notebook to your repository.